# Indian Sign Language Gesture Recognition
## Complete Training Pipeline

This notebook trains a 3D CNN model on ISL gesture videos

## Setup and Imports

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras.callbacks import ModelCheckpoint
import seaborn as sns

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Add paths
sys.path.append('feature_extraction')
sys.path.append('models')
sys.path.append('mediapipe')

import feature_extraction as fe
import model as m

print("✅ Imports successful!")

## Step 1: Extract Features from Videos

In [ ]:
# Check if features already exist
if os.path.exists('X.npy') and os.path.exists('y.npy'):
    print("Loading pre-extracted features...")
    X = np.load('X.npy')
    y = np.load('y.npy')
    print(f"✅ Loaded features:")
    print(f"   X shape: {X.shape}")
    print(f"   y shape: {y.shape}")
else:
    print("Extracting features from videos (this may take 5-10 minutes)...")
    X, y = fe.combine_features()
    print(f"✅ Features extracted successfully!")

## Step 2: Analyze Gesture Classes

In [ ]:
# Get unique gestures
gestures = sorted(np.unique(y))
print(f"Found {len(gestures)} gesture classes:")
print()

class_counts = {}
for gesture in gestures:
    count = np.sum(y == gesture)
    class_counts[gesture] = count
    print(f"  {gesture:10s}: {count:3d} videos")

# Plot class distribution
plt.figure(figsize=(10, 6))
plt.bar(range(len(gestures)), [class_counts[g] for g in gestures], color='steelblue')
plt.xlabel('Gesture Class')
plt.ylabel('Number of Videos')
plt.title('Class Distribution - ISL Gestures')
plt.xticks(range(len(gestures)), gestures, rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nTotal videos: {len(y)}")

## Step 3: Split Data into Train/Validation/Test Sets

In [ ]:
# First split: train+val (80%) vs test (20%)
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Second split: train (90%) vs val (10%) of remaining
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.1, stratify=y_train_val, random_state=42
)

print("Data Split Summary:")
print(f"  Training set:   {X_train.shape[0]:3d} videos ({X_train.shape[0]/len(y)*100:.1f}%)")
print(f"  Validation set: {X_val.shape[0]:3d} videos ({X_val.shape[0]/len(y)*100:.1f}%)")
print(f"  Test set:       {X_test.shape[0]:3d} videos ({X_test.shape[0]/len(y)*100:.1f}%)")
print(f"\nData shape: {X_train.shape}")
print(f"  - Frames per video: 30")
print(f"  - Frame size: 224x224 pixels")
print(f"  - Channels: 3 (RGB)")

## Step 4: Build 3D CNN Model

In [ ]:
num_classes = len(gestures)
input_shape = (30, 224, 224, 3)  # (frames, height, width, channels)

print(f"Building model for {num_classes} gesture classes...\n")

model = m.build_spatiotemporal_cnn(input_shape=input_shape, num_classes=num_classes)

print("\nModel Architecture:")
model.summary()

## Step 5: Train the Model

In [ ]:
# Create callbacks
callbacks = m.get_callbacks() + [
    ModelCheckpoint(
        'isl_gesture_model_best.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

print("Starting training...\n")

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=4,
    callbacks=callbacks,
    verbose=1
)

print("\n✅ Training complete!")

## Step 6: Plot Training History

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
axes[0].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2, marker='.')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2, marker='.')
axes[0].set_title('Model Accuracy Over Epochs', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss plot
axes[1].plot(history.history['loss'], label='Train Loss', linewidth=2, marker='.')
axes[1].plot(history.history['val_loss'], label='Val Loss', linewidth=2, marker='.')
axes[1].set_title('Model Loss Over Epochs', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Training history plot saved as 'training_history.png'")

## Step 7: Save Model

In [ ]:
# Save final model
model.save('isl_gesture_model_final.h5')
print("✅ Model saved as 'isl_gesture_model_final.h5'")

# Save model info
import json
model_info = {
    'num_classes': num_classes,
    'gestures': gestures,
    'input_shape': input_shape
}
with open('model_info.json', 'w') as f:
    json.dump(model_info, f, indent=2)
print("✅ Model info saved as 'model_info.json'")

## Step 8: Evaluate on Test Set

In [ ]:
print("Evaluating model on test set...\n")

test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)

print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

## Step 9: Detailed Classification Report

In [ ]:
# Get predictions
y_pred_probs = model.predict(X_test, verbose=0)
y_pred = y_pred_probs.argmax(axis=1)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=gestures, digits=4))

## Step 10: Confusion Matrix

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=gestures, yticklabels=gestures,
            cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix - ISL Gesture Recognition', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Confusion matrix saved as 'confusion_matrix.png'")

## Summary and Results

In [ ]:
print("="*80)
print("TRAINING SUMMARY - ISL GESTURE RECOGNITION")
print("="*80)
print(f"\nModel Architecture: 3D CNN with Batch Normalization and Dropout")
print(f"\nGesture Classes ({num_classes}):")
for i, gesture in enumerate(gestures):
    print(f"  {i}: {gesture}")

print(f"\nDataset Statistics:")
print(f"  Training samples:   {X_train.shape[0]}")
print(f"  Validation samples: {X_val.shape[0]}")
print(f"  Test samples:       {X_test.shape[0]}")
print(f"  Total samples:      {len(y)}")

print(f"\nModel Performance:")
print(f"  Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"  Test Loss:     {test_loss:.4f}")

print(f"\nSaved Files:")
print(f"  - isl_gesture_model_final.h5 (Final model)")
print(f"  - isl_gesture_model_best.h5 (Best model during training)")
print(f"  - model_info.json (Model configuration)")
print(f"  - training_history.png (Accuracy and loss plots)")
print(f"  - confusion_matrix.png (Confusion matrix visualization)")

print("\n" + "="*80)
print("✅ Training pipeline completed successfully!")
print("="*80)